In [1]:
#| default_exp machine_learning.fastai_bridge

> **AI-generated — pending human review.** Created by Claude Code on 2026-10-21. Review checklist: `_nbdev_development/AI_REVIEW_CHECKLIST.md` (workspace root). Remove this cell only after review passes.

fastai `.pkl` learners (note types, note division) break in three ways that every consumer used to patch around inline:

1. Pickles saved under fastai < 2.8 reference `fastcore.transform`/`fastcore.dispatch`, which moved to the separate `fasttransform` package — naive `load_learner` crashes with an opaque import error.
2. A learner pickled on Linux carries a `PosixPath`; loading on Windows needs the `pathlib.PosixPath = pathlib.WindowsPath` swap ([StackOverflow](https://stackoverflow.com/questions/57286486/i-cant-load-my-model-because-i-cant-put-a-posixpath)).
3. A missing `.pkl` surfaces as a raw unpickling failure instead of "classifier unavailable".

`load_fastai_learner` handles all three in one place, **fails loudly** where the ad-hoc playground shims failed silently (no fallback to `fastcore.basics`, which loads the wrong module when `fasttransform` is absent), and keeps `fastai` a *lazy* import so this module — unlike an exported `from fastai... import ...` — never makes its importer depend on fastai being installed.

In [2]:
#| export
import os
import sys
from os import PathLike
from pathlib import Path

In [3]:
#| export
def _install_fastcore_transform_shim() -> None:
    """
    Alias `fastcore.transform`/`fastcore.dispatch` to `fasttransform`.

    Needed to unpickle learners saved under fastai < 2.8, where these
    modules lived inside `fastcore`. Raises `ModuleNotFoundError` with an
    actionable message when the `fasttransform` package is absent — never
    falls back to a wrong module.
    """
    try:
        import fasttransform
    except ModuleNotFoundError as e:
        raise ModuleNotFoundError(
            'Loading fastai learners pickled under fastai < 2.8 requires '
            'the `fasttransform` package: pip install fasttransform') from e
    sys.modules['fastcore.transform'] = fasttransform
    sys.modules['fastcore.dispatch'] = fasttransform

In [4]:
#| export
def load_fastai_learner(path: PathLike, cpu: bool = True):
    """
    Load a fastai learner `.pkl` on any OS, failing loudly and clearly.

    Applies the `fasttransform` shim (see `_install_fastcore_transform_shim`)
    and the `PosixPath` → `WindowsPath` swap needed to load a learner
    pickled on another platform. Missing file raises `FileNotFoundError`
    and missing support package `ModuleNotFoundError`, so every consumer
    gets one uniform 'classifier unavailable' path instead of an opaque
    unpickling crash. `fastai` is imported lazily, so importing this
    module never requires it.

    **Parameters**
    - `path` (*PathLike*) — The `.pkl` written by fastai's `save`.
    - `cpu` (*bool*) — Load onto CPU (default; these learners are inference-only).
    """
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'fastai learner not found at {path}.')
    _install_fastcore_transform_shim()
    import pathlib
    from fastai.learner import load_learner
    restore = None
    if os.name == 'nt':
        restore = pathlib.PosixPath
        pathlib.PosixPath = pathlib.WindowsPath
    try:
        return load_learner(path, cpu=cpu)
    finally:
        if restore is not None:
            pathlib.PosixPath = restore

In [5]:
import pathlib
from unittest import mock
from fastcore.test import *

In [6]:
# Missing file fails with FileNotFoundError before touching fastai/shims.
ExceptionExpected(lambda: load_fastai_learner(path='does_not_exist.pkl'))

# With `fasttransform` unavailable, the shim raises the actionable error
# (patched into sys.modules so this holds whether or not it is installed).
with mock.patch.dict(sys.modules, {'fastcore.transform': None,
                                   'fastcore.dispatch': None,
                                   'fasttransform': None}):
    ExceptionExpected(_install_fastcore_transform_shim)

# The shim aliases both moved module names when fasttransform imports.
fake = mock.MagicMock()
with mock.patch.dict(sys.modules, {'fasttransform': fake,
                                   'fastcore.transform': None,
                                   'fastcore.dispatch': None}):
    sys.modules.pop('fastcore.transform')
    sys.modules.pop('fastcore.dispatch')
    _install_fastcore_transform_shim()
    test_is(sys.modules['fastcore.transform'], fake)
    test_is(sys.modules['fastcore.dispatch'], fake)